# Creador de actas de reuniones

En este Colab, creamos un programa de actas de reuniones.

Incluye código útil para conectar tu Google Drive a tu Colab.

¡Sube tu propio audio para que esto funcione de lujo!

https://colab.research.google.com/drive/1YPfRqoo4UuF-CJlMN7RL2WLi-kRpO6OS?usp=share_link

Esto debería funcionar bien en una caja T4 de bajo costo o gratuita.

# Crear actas de reuniones a partir de un archivo de audio

Descargué algunas actas de reuniones del Ayuntamiento de Denver de este conjunto de datos:
https://huggingface.co/datasets/huuuyeah/meetingbank

Las coloqué en mi Google Drive. El objetivo de este producto es utilizar el audio para generar actas de reuniones, incluidas las acciones.

Para que esto funcione, también deberá descargar un clip del Ayuntamiento de Denver o, para más diversión, ¡grabe algo propio!

In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate openai

In [ ]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
# AutoModelForSpeechSeq2Seq, AutoProcessor y pipeline los usa la celda de la implementación
# alternativa con Whisper open source del final: faltaban en los imports originales.
from transformers import (AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig,
                          AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline)
import torch

In [ ]:
# Constants

AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [ ]:
# Nueva habilidad: conectar este Colab a mi Google Drive

drive.mount("/content/drive")
audio_filename = "/content/denver_extract.mp3"

# Archivo de audio

Puedes usar el mismo archivo que yo, el extracto de las actas del ayuntamiento de Denver, o puedes probar con uno propio.

Si quieres usar el mismo que yo, descarga mi extracto aquí y colócalo en tu Google Drive:
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

In [ ]:
# Conectate a HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
#Iniciar sesión en OpenAI usando Secrets en Colab

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

In [ ]:
# Usa el modelo Whisper OpenAI para convertir el audio en texto
# Si prefieres usar un modelo de código abierto, el estudiante de la clase
# Youssef ha contribuido con una versión de código abierto
# que he agregado al final de este Colab

audio_file = open(audio_filename, "rb")
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
system_message = "Eres un asistente que produce actas de reuniones a partir de transcripciones, con resumen, puntos clave de discusión, conclusiones y elementos de acción con los propietarios, en formato Markdown. Aunque el audio original esté en inglés, quiero que me proporciones el acta en español."
user_prompt = f"A continuación, se incluye un extracto de la transcripción de una reunión del consejo de Denver. Escribe las actas traducidas al castellano en formato Markdown, incluyendo un resumen con los asistentes, la ubicación y la fecha; puntos de discusión; conclusiones; y elementos de acción con los propietarios.\n{transcription}"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
# `apply_chat_template` ya no devuelve un tensor plano sino un BatchEncoding: hay que pedirle
# `return_dict=True` y desempaquetarlo con ** al llamar a generate(), o generate() revienta
# con un AttributeError sin mensaje.
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True).to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(**inputs, max_new_tokens=2000, streamer=streamer)

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))

## Implementación alternativa

El alumno de la clase Youssef ha contribuido con esta variación en la que utilizamos un modelo de código abierto para transcribir el audio de la reunión.

¡Gracias Youssef!

In [ ]:
AUDIO_MODEL = "openai/whisper-medium"
speech_model = AutoModelForSpeechSeq2Seq.from_pretrained(AUDIO_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True, use_safetensors=True)
speech_model.to('cuda')
processor = AutoProcessor.from_pretrained(AUDIO_MODEL)

pipe = pipeline(
    "automatic-speech-recognition",
    model=speech_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch.float16,
    device='cuda',
)

In [ ]:
# Utiliza el modelo Whisper OpenAI para convertir el audio en texto
result = pipe(audio_filename)

In [ ]:
transcription = result["text"]
print(transcription)